# STaR SFT — Qwen3.8-27B (Google Colab version)

Same training as `star_sft_qwen38.ipynb`, adapted to run on Colab instead of the local RTX 3090
(which doesn't have enough VRAM for this model at the sequence lengths this dataset needs — verified).

**Before running, prepare two things:**
1. Your Kaggle API token (`kaggle.json`) — needed to download the 27B model (~30GB) from the Kaggle
   dataset `driessmit1/qwen3-8-27b-fp8-hf-017b9c7a`. Get it from kaggle.com → Account → Create New API Token.
2. Upload `star_sft_qwen38_20260923.jsonl` (the already-mined 73MB dataset, 2753 turns) to your Google
   Drive, e.g. at `MyDrive/arc-agi-3/star_sft_qwen38_20260923.jsonl`. No need to re-run the mining step.

**Runtime → Change runtime type → GPU.** Colab Pro doesn't guarantee A100 — Cell 2 below checks your
actual VRAM and picks a safe config automatically (or tells you clearly if the GPU you got won't fit).

In [ ]:
# ── Cell 1: Mount Drive, set up Kaggle credentials ───────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/arc-agi-3')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

DATASET_JSONL = DRIVE_DIR / 'star_sft_qwen38_20260923.jsonl'
if not DATASET_JSONL.exists():
    from google.colab import files
    print(f'{DATASET_JSONL} not found on Drive -- upload star_sft_qwen38_20260923.jsonl now '
          '(pick the file from your local machine):')
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError('No file uploaded. Re-run this cell and pick the .jsonl file.')
    with open(DATASET_JSONL, 'wb') as f:
        f.write(list(uploaded.values())[0])
    print(f'Saved to {DATASET_JSONL} -- will be found automatically on Drive in future sessions.')
print(f'Dataset found: {DATASET_JSONL} ({DATASET_JSONL.stat().st_size / 1e6:.1f} MB)')

# Kaggle auth: newer kaggle CLI versions use a single opaque ~/.kaggle/access_token
# file instead of the classic username+key kaggle.json -- support both. Check which
# one your local machine has: `ls ~/.kaggle/` (access_token vs kaggle.json).
os.makedirs('/root/.kaggle', exist_ok=True)
KAGGLE_JSON_DRIVE  = DRIVE_DIR / 'kaggle.json'
KAGGLE_TOKEN_DRIVE = DRIVE_DIR / 'access_token'

if KAGGLE_JSON_DRIVE.exists():
    import shutil
    shutil.copy(KAGGLE_JSON_DRIVE, '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print('Kaggle credentials ready (kaggle.json).')
elif KAGGLE_TOKEN_DRIVE.exists():
    import shutil
    shutil.copy(KAGGLE_TOKEN_DRIVE, '/root/.kaggle/access_token')
    os.chmod('/root/.kaggle/access_token', 0o600)
    print('Kaggle credentials ready (access_token).')
else:
    from google.colab import files
    print('No Kaggle credentials found on Drive. Upload EITHER kaggle.json (kaggle.com -> Account -> '
          'Create New API Token) OR your local ~/.kaggle/access_token file now:')
    uploaded = files.upload()
    fname, content = list(uploaded.items())[0]
    target = '/root/.kaggle/kaggle.json' if fname.endswith('.json') else '/root/.kaggle/access_token'
    with open(target, 'wb') as f:
        f.write(content)
    os.chmod(target, 0o600)
    print(f'Saved to {target}.')

In [ ]:
# ── Cell 2: GPU check -- pick a safe config, or abort early before wasting compute units ──
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU attached. Runtime -> Change runtime type -> GPU, then re-run.')

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {gpu_name}  VRAM: {vram_gb:.1f} GB')

# Verified on a 24GB RTX 3090 (real end-to-end test, 2026-09-23): with fla installed +
# logits_to_keep + attn-only LoRA + empty_cache, still OOM'd by ~150-300MB regardless of
# length (5120-8192) or LoRA config -- the irreducible cost is a bitsandbytes NF4->bf16
# dequant buffer (~150-200MB) needed per MLP layer during gradient-checkpoint recompute.
# So: <26GB is NOT safe (would hit the same wall). >=32GB has enough real margin for the
# full config (real prompts run 5000-6400 tokens, full sequences up to ~11500).
if vram_gb < 26:
    raise RuntimeError(
        f'{gpu_name} ({vram_gb:.1f}GB) will hit the same OOM wall as the local RTX 3090 '
        '(verified: this model needs >26GB in practice, not just on paper). '
        'In Colab: Runtime -> Change runtime type -> make sure a bigger GPU (A100/L4-24GB+) is selected, '
        'or use Colab Pro+ for guaranteed A100. Do not just retry -- it will OOM the same way.'
    )
elif vram_gb < 34:
    print('Tight-ish GPU (24-34GB) -- using the constrained config validated on the 3090.')
    MAX_LENGTH = 5632
    LORA_TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj']
else:
    print('Comfortable VRAM (34GB+) -- using the full config (better dataset coverage + quality).')
    MAX_LENGTH = 8192
    LORA_TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

print(f'MAX_LENGTH={MAX_LENGTH}  LORA_TARGET_MODULES={LORA_TARGET_MODULES}')

In [ ]:
# ── Cell 3: Install/upgrade packages ─────────────────────────────────────────
# Colab's preinstalled transformers is usually too old for the Qwen3_5 architecture,
# and flash-linear-attention (fla) is required -- without it this model's linear_attn
# layers fall back to a pure-PyTorch kernel that OOMs almost immediately (verified).
!pip install -q -U transformers accelerate peft bitsandbytes flash-linear-attention kaggle

In [ ]:
# ── Cell 4: Download Qwen3.8-27B FP8 from Kaggle ─────────────────────────────
from pathlib import Path
import subprocess

MODEL_DIR = Path('/content/qwen38_27b_fp8')

if MODEL_DIR.exists() and any(MODEL_DIR.glob('*.safetensors')):
    shard_count = len(list(MODEL_DIR.glob('*.safetensors')))
    total_gb = sum(p.stat().st_size for p in MODEL_DIR.glob('*.safetensors')) / 1e9
    print(f'Model already present: {shard_count} shards, {total_gb:.1f} GB')
else:
    print('Downloading Qwen3.8-27B FP8 (~30GB, this takes a while)...')
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ['kaggle', 'datasets', 'download',
         'driessmit1/qwen3-8-27b-fp8-hf-017b9c7a',
         '-p', str(MODEL_DIR), '--unzip'],
        check=True
    )
    shard_count = len(list(MODEL_DIR.glob('**/*.safetensors')))
    print(f'Download complete. Shards: {shard_count}')

In [ ]:
# ── Cell 4b: Dequantize FP8 -> BF16 (CRITICAL FIX, 2026-09-24) ───────────────
# The raw checkpoint stores weights as float8_e4m3fn with a companion
# weight_scale_inv tensor per 128x128 block -- true_value = fp8_value * block_scale.
# The old approach (delete quantization_config, cast straight to bf16) SILENTLY
# SKIPS this rescaling. Verified on the local RTX 3090: the resulting model loads
# with no NaN but generates pure gibberish even on "Hello, how are you?" (foreign-
# script garbage tokens). The ~23h Colab run before this fix trained a LoRA on
# top of these mis-scaled weights -- that checkpoint is worthless, training
# restarts from scratch below with a NEW adapter dir so it can't accidentally resume.
import json, gc
import torch
from pathlib import Path
from safetensors import safe_open
from safetensors.torch import save_file
from transformers import AutoConfig as _AC, AutoTokenizer as _AT

BF16_DIR = Path('/content/qwen38_27b_bf16')

if BF16_DIR.exists() and any(BF16_DIR.glob('*.safetensors')):
    print(f'Dequantized BF16 model already present: {BF16_DIR}')
else:
    print('Dequantizing FP8 -> BF16 (~54GB disk, a few minutes)...')
    BF16_DIR.mkdir(parents=True, exist_ok=True)

    def _find_root(base):
        if (base / 'config.json').exists():
            return base
        for p in sorted(base.rglob('config.json')):
            return p.parent
        raise FileNotFoundError

    src = _find_root(MODEL_DIR)
    idx = json.loads((src / 'model.safetensors.index.json').read_text())
    weight_map = idx['weight_map']
    opened = {s: safe_open(str(src / s), framework='pt', device='cpu') for s in set(weight_map.values())}

    def get_tensor(key):
        shard = weight_map.get(key)
        return opened[shard].get_tensor(key) if shard else None

    def dequantize_fp8(w_fp8, scale):
        if scale.ndim == 0:
            return (w_fp8.to(torch.float32) * scale.to(torch.float32)).to(torch.bfloat16)
        while scale.ndim > 2:
            scale = scale.squeeze(0)
        if scale.ndim == 2:
            w_f32 = w_fp8.to(torch.float32)
            rows, cols = w_f32.shape[-2:]
            sr, sc = scale.shape
            bm, bn = rows // sr, cols // sc
            s_f32 = scale.to(torch.float32)
            q = w_f32.reshape(sr, bm, sc, bn)
            s = s_f32.unsqueeze(1).unsqueeze(3)
            return (q * s).to(torch.bfloat16).reshape(rows, cols)
        return (w_fp8.to(torch.float32) * scale.to(torch.float32)).to(torch.bfloat16)

    FP8_DTYPES = {torch.float8_e4m3fn, getattr(torch, 'float8_e5m2', None)}
    SKIP_KEYS = {'weight_scale_inv', 'activation_scale'}
    export_index = {'metadata': {'format': 'pt'}, 'weight_map': {}}
    current_shard, current_bytes, shard_idx = {}, 0, 0
    all_keys = sorted(weight_map.keys())
    print(f'{len(all_keys)} tensors')

    for i, key in enumerate(all_keys):
        if any(s in key for s in SKIP_KEYS):
            continue
        tensor = get_tensor(key)
        if tensor is None:
            continue
        if tensor.dtype in FP8_DTYPES:
            scale = get_tensor(key.replace('.weight', '.weight_scale_inv'))
            tensor = dequantize_fp8(tensor, scale) if scale is not None else tensor.to(torch.bfloat16)
        elif tensor.dtype not in (torch.bfloat16, torch.float16):
            tensor = tensor.to(torch.bfloat16)
        tensor = tensor.contiguous()
        shard_name = f'model-shard-{shard_idx:05d}.safetensors'
        current_shard[key] = tensor
        export_index['weight_map'][key] = shard_name
        current_bytes += tensor.numel() * tensor.element_size()
        if current_bytes >= 4 * 1024**3:
            print(f'  [{i+1}/{len(all_keys)}] writing shard {shard_idx} ({current_bytes/1024**3:.1f}GB)')
            save_file(current_shard, str(BF16_DIR / shard_name))
            current_shard, current_bytes = {}, 0
            shard_idx += 1
            gc.collect()
    if current_shard:
        save_file(current_shard, str(BF16_DIR / f'model-shard-{shard_idx:05d}.safetensors'))

    (BF16_DIR / 'model.safetensors.index.json').write_text(json.dumps(export_index, indent=2))
    cfg = _AC.from_pretrained(str(src), trust_remote_code=True)
    if hasattr(cfg, 'quantization_config'):
        cfg.quantization_config = None
    cfg.save_pretrained(str(BF16_DIR))
    _AT.from_pretrained(str(src)).save_pretrained(str(BF16_DIR))
    print(f'Dequantization complete: {BF16_DIR}')

MODEL_DIR = BF16_DIR  # Cell 5 loads from here now

In [ ]:
# ── Cell 5: Load Qwen3.8 (corrected BF16) with BnB NF4 ───────────────────────
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig

def _find_model_root(base: Path) -> Path:
    if (base / 'config.json').exists():
        return base
    for p in sorted(base.rglob('config.json')):
        return p.parent
    raise FileNotFoundError(f'config.json not found under {base}')

MODEL_ROOT = _find_model_root(MODEL_DIR)
print(f'Loading from: {MODEL_ROOT}')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_ROOT))
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Cell 4b already produced a real dequantized BF16 checkpoint with
# quantization_config removed -- nothing to strip here.
cfg = AutoConfig.from_pretrained(str(MODEL_ROOT), trust_remote_code=True)

print('Loading model (NF4)...')
model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_ROOT),
    config=cfg,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    attn_implementation='sdpa',
)

free_gb = torch.cuda.mem_get_info()[0] / 1024**3
print(f'Model loaded. VRAM free: {free_gb:.1f} GB')

# Sanity check -- isnan() alone is NOT enough (verified: mis-scaled FP8 weights
# loaded without NaN but generated pure gibberish). Actually read the text.
dummy = tokenizer.apply_chat_template(
    [{'role': 'user', 'content': 'Hello, how are you? Answer in one short sentence.'}],
    tokenize=False, add_generation_prompt=True,
)
dummy_inputs = tokenizer(dummy, return_tensors='pt').to(model.device)
with torch.no_grad():
    dummy_out = model.generate(**dummy_inputs, max_new_tokens=30, do_sample=False,
                                pad_token_id=tokenizer.eos_token_id)
sanity_text = tokenizer.decode(dummy_out[0][dummy_inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f'Sanity generation: {sanity_text!r}')
print('^ must be coherent English -- if it looks like gibberish/foreign-script tokens, '
      'STOP, do not train, the checkpoint is mis-scaled again.')

In [ ]:
# ── Cell 6: Prepare dataset for SFT ──────────────────────────────────────────
import json, random
from torch.utils.data import Dataset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)

records = [json.loads(l) for l in open(DATASET_JSONL) if l.strip()]
random.shuffle(records)

class StarSFTDataset(Dataset):
    def __init__(self, records, tokenizer, max_length):
        self.samples = []
        skipped = 0
        for r in tqdm(records, desc='Tokenizing', unit='sample'):
            messages = [
                {'role': 'system',    'content': r['system']},
                {'role': 'user',      'content': r['user']},
                {'role': 'assistant', 'content': r['assistant']},
            ]
            try:
                full_text = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=False
                )
                prompt_text = tokenizer.apply_chat_template(
                    messages[:-1], tokenize=False, add_generation_prompt=True
                )
            except Exception:
                skipped += 1
                continue

            enc = tokenizer(full_text, truncation=True, max_length=max_length, return_tensors='pt')
            if enc['input_ids'].shape[1] < 20:
                skipped += 1
                continue

            prompt_len = tokenizer(
                prompt_text, truncation=True, max_length=max_length, return_tensors='pt'
            )['input_ids'].shape[1]

            ids    = enc['input_ids'][0]
            attn   = enc['attention_mask'][0]
            labels = ids.clone()
            labels[:prompt_len] = -100

            if (labels != -100).sum().item() == 0:
                skipped += 1
                continue

            self.samples.append({'input_ids': ids, 'attention_mask': attn, 'labels': labels})

        print(f'Dataset: {len(self.samples)} samples ({skipped} skipped)')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


dataset = StarSFTDataset(records, tokenizer, MAX_LENGTH)
lens = [s['input_ids'].shape[0] for s in dataset]
print(f'Token lengths: min={min(lens)} median={sorted(lens)[len(lens)//2]} max={max(lens)}')
truncated = sum(1 for l in lens if l == MAX_LENGTH)
print(f'Truncated at {MAX_LENGTH}: {truncated}/{len(lens)}')

In [ ]:
# ── Cell 7: Configure LoRA ────────────────────────────────────────────────────
from peft import LoraConfig, get_peft_model, TaskType

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=LORA_TARGET_MODULES,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ── Cell 8: Train ─────────────────────────────────────────────────────────────
import torch
import torch.nn.functional as F
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

# New dir name (not star_qwen38_adapter): the old one holds a checkpoint trained
# on the mis-scaled FP8-as-bf16 base model (worthless, see Cell 4b). Using a new
# name guarantees this cell can't accidentally resume from that bad checkpoint.
# The old checkpoint on Drive can be deleted manually to free space.
ADAPTER_DIR = DRIVE_DIR / 'star_qwen38_adapter_v2'
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

_checkpoints = sorted(
    ADAPTER_DIR.glob('checkpoint-*'),
    key=lambda p: int(p.name.split('-')[-1])
)
RESUME_FROM = str(_checkpoints[-1]) if _checkpoints else None
if RESUME_FROM:
    print(f'Found checkpoint, resuming from: {RESUME_FROM}')
else:
    print('No checkpoint found, starting fresh.')

n_eval = max(1, len(dataset) // 10)
train_ds = torch.utils.data.Subset(dataset, range(len(dataset) - n_eval))
eval_ds  = torch.utils.data.Subset(dataset, range(len(dataset) - n_eval, len(dataset)))
print(f'Train: {len(train_ds)}  Eval: {len(eval_ds)}')

collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, padding=True, pad_to_multiple_of=8, label_pad_token_id=-100,
)

# logits_to_keep: never project the 151936-vocab LM head over the masked prompt tokens
# (~80%+ of every sequence) -- projecting the full sequence is what made this OOM even
# on a 24GB card. Keeping this even on a bigger GPU costs nothing and trains faster.
class BF16LossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        seq_len = labels.shape[1]
        valid_mask = labels[0] != -100
        nz = valid_mask.nonzero()

        if nz.numel() == 0:
            outputs = model(**inputs, logits_to_keep=1)
            loss = outputs.logits.sum() * 0.0
            return (loss, outputs) if return_outputs else loss

        first_valid = int(nz[0].item())
        logits_to_keep = seq_len - first_valid + 1

        outputs = model(**inputs, logits_to_keep=logits_to_keep)
        logits = outputs.logits
        shift_logits = logits[:, :-1, :].float()
        shift_labels = labels[:, first_valid:].to(shift_logits.device)

        loss = F.cross_entropy(
            shift_logits.reshape(-1, shift_logits.size(-1)),
            shift_labels.reshape(-1),
            ignore_index=-100,
        )

        if return_outputs:
            return loss, outputs

        del outputs, logits, shift_logits
        torch.cuda.empty_cache()
        return loss


# EPOCHS=1 first (not 3): Colab/Kaggle GPU-hours are scarce after the wasted
# ~23h run on the mis-scaled checkpoint. ~7-8h instead of ~23h -- gives an early
# read on whether STaR SFT helps at all before committing the full budget.
# Bump back to 3 only if 1-epoch results look promising.
EPOCHS   = 1
BATCH    = 1
GRAD_ACC = 8
WARMUP_STEPS = max(1, int(len(train_ds) / GRAD_ACC * EPOCHS * 0.05))

args = TrainingArguments(
    output_dir=str(ADAPTER_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=WARMUP_STEPS,
    fp16=False,
    bf16=True,
    logging_steps=5,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='epoch',
    save_total_limit=1,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim='paged_adamw_8bit',
    report_to='none',
    seed=SEED,
    dataloader_pin_memory=False,
)

trainer = BF16LossTrainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds, data_collator=collator,
)

print(f'warmup_steps={WARMUP_STEPS}  Starting STaR SFT training...')
trainer.train(resume_from_checkpoint=RESUME_FROM)
print('Training complete.')

In [ ]:
# ── Cell 9: Save adapter to Drive (persists after the Colab session ends) ────
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print(f'Adapter saved -> {ADAPTER_DIR}')

for p in sorted(ADAPTER_DIR.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size // 1024} KB)')

model.eval()
prompt = tokenizer.apply_chat_template(
    [{'role': 'system', 'content': 'You are a coding agent solving a grid-based puzzle game.'},
     {'role': 'user',   'content': 'Level 1, action 0. Board:\nBBBBBB\nB....B\nB....B\nBBBBBB\n\nAnalyse and act.'}],
    tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
resp = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f'\nSample response (first 500):\n{resp[:500]}')

## Next steps after training

1. **Download the adapter from Drive** (`MyDrive/arc-agi-3/star_qwen38_adapter/`) to the local repo at
   `checkpoints/star_qwen38_YYYYMMDD/`, or upload straight from Drive to a new Kaggle dataset:
   ```bash
   kaggle datasets create -p <local_adapter_dir> -t "STaR SFT Qwen3.8 adapter"
   ```
2. **Switch kernel model** in `notebooks/build_tuning_variant.py`:
   - Replace `driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot` -> `driessmit1/qwen3-8-27b-fp8-hf-017b9c7a`
   - Add the adapter dataset
   - Add a LEVER that loads the adapter at inference time (Cell 13 hook)